# PIEP Modern Archived Examples

This notebook is a notebook-oriented walkthrough of the archived CuPc, GRGDS, and Lysozyme examples.
It is not meant to mirror the legacy command transcript line by line. Instead it follows the same scientific path described in `pub.md`:

1. start from SAD observations
2. prepare and classify those observations
3. run the GM search
4. conventionalize the best reduced cells

The modern Python API keeps those steps explicit while hiding the legacy 18-field bookkeeping.

In [ ]:
import piep
import piep.examples as examples

## Search And Conventionalization

In the publication, the scientific question is not just whether a candidate cell can be generated, but whether the full workflow ends in the right lattice description. The helper below runs the full modern path from archived SAD patterns to the preferred conventional cell.

The important outputs to inspect are:

- the total GM candidate count
- the top search candidates and their figures of merit
- the preferred conventional setting chosen from the reduced cell

In [ ]:
for case in examples.archived_cases():
    result = piep.search_and_conventionalize(
        case.patterns,
        volume_min=case.volume_min,
        volume_max=case.volume_max,
        centering=case.centering,
        preferred_centering=case.preferred_centering,
        minimum_system=case.minimum_system,
        top_n=3,
    )
    preferred = result.preferred_candidate.preferred_candidate
    print(case.name)
    print('  total GM candidates :', result.search.total_candidate_count)
    print('  top FOM             :', result.search.candidates[0].figure_of_merit)
    print('  preferred setting   :', preferred.system, preferred.centering, preferred.cell)
    print()

## Pattern-Level Indexing

The publication also reasons in terms of individual SAD patterns and the reflection pairs that explain them. The high-level API exposes that path through `piep.index_pattern(...)`. This is useful for debugging, for synthetic-data generation, and for relating the final cell back to specific indexed patterns.

In [ ]:
cupc = examples.cupc_case()
indexed = piep.index_pattern(cupc.patterns[0].observation, cupc.cell, centering=cupc.centering)
top = indexed.top_match
print('Top CuPc match:')
print('  first hkl :', top.first_hkl)
print('  second hkl:', top.second_hkl)
print('  zone axis :', top.zone_axis)
print('  FOM       :', top.figure_of_merit)

## Interpretation

A useful way to read these results is:

- the archived SAD patterns remain the transcript-tight oracle
- the search result tells us whether the modern GM search is looking in the right part of lattice space
- the conventionalization result tells us whether the modern post-processing is translating the reduced cell into the scientifically expected setting

That separation mirrors the scientific logic in the publication and keeps the modern code easy to audit.